In [1]:
import pandas as pd 
df = pd.read_csv("Data/ner.csv")
df.head()

,Sentence #,Sentence,POS,Tag
0,Sentence: 1,Thousands of demonstrators have marched throug...,"['NNS', 'IN', 'NNS', 'VBP', 'VBN', 'IN', 'NNP'...","['O', 'O', 'O', 'O', 'O', 'O', 'B-geo', 'O', '..."
1,Sentence: 2,Families of soldiers killed in the conflict jo...,"['NNS', 'IN', 'NNS', 'VBN', 'IN', 'DT', 'NN', ...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
2,Sentence: 3,They marched from the Houses of Parliament to ...,"['PRP', 'VBD', 'IN', 'DT', 'NNS', 'IN', 'NN', ...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
3,Sentence: 4,"Police put the number of marchers at 10,000 wh...","['NNS', 'VBD', 'DT', 'NN', 'IN', 'NNS', 'IN', ...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
4,Sentence: 5,The protest comes on the eve of the annual con...,"['DT', 'NN', 'VBZ', 'IN', 'DT', 'NN', 'IN', 'D...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."


In [2]:
df.shape

(47959, 4)

In [3]:
df.drop(columns = ['POS'] , axis = 1 , inplace = True)
df.head()

,Sentence #,Sentence,Tag
0,Sentence: 1,Thousands of demonstrators have marched throug...,"['O', 'O', 'O', 'O', 'O', 'O', 'B-geo', 'O', '..."
1,Sentence: 2,Families of soldiers killed in the conflict jo...,"['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
2,Sentence: 3,They marched from the Houses of Parliament to ...,"['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
3,Sentence: 4,"Police put the number of marchers at 10,000 wh...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
4,Sentence: 5,The protest comes on the eve of the annual con...,"['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."


In [4]:
from sklearn.model_selection import train_test_split
train_data , test_data = train_test_split(df , test_size = 0.3 , random_state = 42)
test_data , validation_data = train_test_split(test_data , test_size = 0.5 , random_state = 42)

In [5]:
train_data.shape , test_data.shape , validation_data.shape

((33571, 3), (7194, 3), (7194, 3))

In [6]:
from datasets import Dataset , DatasetDict
dataset = DatasetDict({
    'train' : Dataset.from_pandas(train_data),
    'test': Dataset.from_pandas(test_data),
    'valid': Dataset.from_pandas(validation_data)
})
dataset

DatasetDict({
    train: Dataset({
        features: ['Sentence #', 'Sentence', 'Tag', '__index_level_0__'],
        num_rows: 33571
    })
    test: Dataset({
        features: ['Sentence #', 'Sentence', 'Tag', '__index_level_0__'],
        num_rows: 7194
    })
    valid: Dataset({
        features: ['Sentence #', 'Sentence', 'Tag', '__index_level_0__'],
        num_rows: 7194
    })
})

In [7]:
dataset = dataset.remove_columns(['Sentence #' , '__index_level_0__'])

In [8]:
dataset['train'][0]

{'Sentence': 'The price of oil has nearly doubled over the past year .',
 'Tag': "['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-tim', 'O', 'O']"}

In [9]:
# convert Tag into list
import ast 

def parse_tags(example):
    example['Tag'] = ast.literal_eval(example['Tag'])
    return example

In [10]:
parse_tags(dataset['train'][0])

{'Sentence': 'The price of oil has nearly doubled over the past year .',
 'Tag': ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-tim', 'O', 'O']}

In [11]:
dataset = dataset.map(parse_tags)

Map:   0%|          | 0/33571 [00:00<?, ? examples/s]

Map:   0%|          | 0/7194 [00:00<?, ? examples/s]

Map:   0%|          | 0/7194 [00:00<?, ? examples/s]

In [12]:
dataset['train'][0]

{'Sentence': 'The price of oil has nearly doubled over the past year .',
 'Tag': ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-tim', 'O', 'O']}

### Create Label Mapping

In [13]:
unique_labels = set()

i = 0
for tag_list in dataset['train']['Tag']:
    for tag in tag_list:
        unique_labels.add(tag)

unique_labels

{'B-art',
 'B-eve',
 'B-geo',
 'B-gpe',
 'B-nat',
 'B-org',
 'B-per',
 'B-tim',
 'I-art',
 'I-eve',
 'I-geo',
 'I-gpe',
 'I-nat',
 'I-org',
 'I-per',
 'I-tim',
 'O'}

In [14]:
unique_labels = sorted(list(unique_labels))

In [15]:
label2id = {l : i for i , l in enumerate(unique_labels)}
id2label = {val : key for key , val in label2id.items()}

In [16]:
id2label

{0: 'B-art',
 1: 'B-eve',
 2: 'B-geo',
 3: 'B-gpe',
 4: 'B-nat',
 5: 'B-org',
 6: 'B-per',
 7: 'B-tim',
 8: 'I-art',
 9: 'I-eve',
 10: 'I-geo',
 11: 'I-gpe',
 12: 'I-nat',
 13: 'I-org',
 14: 'I-per',
 15: 'I-tim',
 16: 'O'}

### Split Sentence into words

In [17]:
def split_words(example):
    example['tokens'] = example['Sentence'].split()
    return example

In [18]:
split_words(dataset['train'][0])

{'Sentence': 'The price of oil has nearly doubled over the past year .',
 'Tag': ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-tim', 'O', 'O'],
 'tokens': ['The',
  'price',
  'of',
  'oil',
  'has',
  'nearly',
  'doubled',
  'over',
  'the',
  'past',
  'year',
  '.']}

In [19]:
dataset = dataset.map(split_words)

Map:   0%|          | 0/33571 [00:00<?, ? examples/s]

Map:   0%|          | 0/7194 [00:00<?, ? examples/s]

Map:   0%|          | 0/7194 [00:00<?, ? examples/s]

In [20]:
dataset

DatasetDict({
    train: Dataset({
        features: ['Sentence', 'Tag', 'tokens'],
        num_rows: 33571
    })
    test: Dataset({
        features: ['Sentence', 'Tag', 'tokens'],
        num_rows: 7194
    })
    valid: Dataset({
        features: ['Sentence', 'Tag', 'tokens'],
        num_rows: 7194
    })
})

### Convert Labels into ids

In [21]:
def convert_labels_into_ids(example):
    example['ner_tags'] = [label2id[tag] for tag in example['Tag']]
    return example

In [22]:
convert_labels_into_ids(dataset['train'][0])

{'Sentence': 'The price of oil has nearly doubled over the past year .',
 'Tag': ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-tim', 'O', 'O'],
 'tokens': ['The',
  'price',
  'of',
  'oil',
  'has',
  'nearly',
  'doubled',
  'over',
  'the',
  'past',
  'year',
  '.'],
 'ner_tags': [16, 16, 16, 16, 16, 16, 16, 16, 16, 7, 16, 16]}

In [23]:
dataset = dataset.map(convert_labels_into_ids)

Map:   0%|          | 0/33571 [00:00<?, ? examples/s]

Map:   0%|          | 0/7194 [00:00<?, ? examples/s]

Map:   0%|          | 0/7194 [00:00<?, ? examples/s]

### Tokenization + Alignment

In [24]:
# load the BERT tokenizer
from transformers import BertTokenizerFast

model_checkpoint = "bert-base-cased"
tokenizer = BertTokenizerFast.from_pretrained(model_checkpoint)

In [25]:
def tokenize_and_align_labels(batch):
    # we are getting a batch of rows, so 
    # step-1: now apply tokenizer on our tokens
    tokenized = tokenizer(
        batch['tokens'],
        truncation = True,
        # as our data is already splitted into words(tokens) so tell the tokenizer
        is_split_into_words = True 
    )
    
    # tokenized: a dict contains 3 keys: input_ids, attention_mask, token_type_ids
    
    # step-2: Initialize empty labels list
    labels = []
    
    # step-3: Loop through each examples
    for i , label in enumerate(batch['ner_tags']):
        # i -> batch index and label -> the NER tags for example i
        # get the word ids
        word_ids = tokenized.word_ids(batch_index = i)
        # Tokens: ["[CLS]" , "[word1]" ........ "[word_n]"]
        # word_ids: [None , 0 , 2(tags).........]
        
        # initialize a label list for the i-th sentence
        label_ids = []
        prev_word = None # Tracks the previous word ID to detect subword continuations
        
        for word_id in word_ids:
            # if word_id is None means special token we ignore them in loss calculation
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word: # a new word starts
                if word_id < len(label):
                   label_ids.append(label[word_id])
                else:
                    label_ids.append(-100)
            else: # continuation of a words sub-word
                label_ids.append(-100)
            
            prev_word = word_id
        
        labels.append(label_ids)
    
    tokenized['labels'] = labels 
    return tokenized

In [26]:
print(tokenizer(
    dataset['train'][0]['tokens'],
    truncation = True,
    is_split_into_words = True
))

{'input_ids': [101, 1109, 3945, 1104, 2949, 1144, 2212, 11590, 1166, 1103, 1763, 1214, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [27]:
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched = True, 
    remove_columns = dataset['train'].column_names
)

Map:   0%|          | 0/33571 [00:00<?, ? examples/s]

Map:   0%|          | 0/7194 [00:00<?, ? examples/s]

Map:   0%|          | 0/7194 [00:00<?, ? examples/s]

In [28]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 33571
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 7194
    })
    valid: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 7194
    })
})

### Define the evaluation function

In [29]:
import numpy as np 
from seqeval.metrics import classification_report, f1_score, precision_score,recall_score,accuracy_score

In [30]:
def compute_metrics(eval_pred):
    predictions , labels = eval_pred
    
    # predictions shape: (batch_size, seq_length, num_labels)
    # Get the index of the max logit
    predictions = np.argmax(predictions , axis = 2)
    
    # removed ignore index means index with value -100
    true_labels = []
    true_predictions = []
    
    for prediction , label in zip(predictions , labels):
        true_label = []
        true_prediction = []
        
        for pred_idx , label_idx in zip(prediction , label):
            if label_idx != -100:
               # Convert index to label string
               true_label.append(id2label[label_idx])
               true_prediction.append(id2label[pred_idx])
        
        true_labels.append(true_label)
        true_predictions.append(true_prediction)
    
    results = {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
        "accuracy": accuracy_score(true_labels, true_predictions),
    }
    return results

In [31]:
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

In [32]:
unique_labels

['B-art',
 'B-eve',
 'B-geo',
 'B-gpe',
 'B-nat',
 'B-org',
 'B-per',
 'B-tim',
 'I-art',
 'I-eve',
 'I-geo',
 'I-gpe',
 'I-nat',
 'I-org',
 'I-per',
 'I-tim',
 'O']

In [33]:
model = AutoModelForTokenClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id
)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [34]:
data_collator = DataCollatorForTokenClassification(tokenizer = tokenizer)

In [35]:
training_args = TrainingArguments(
    output_dir="./ner-model",
    eval_strategy="epoch",  
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    load_best_model_at_end=True,  
    metric_for_best_model="f1"
)

In [36]:
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['valid'],  
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,  
)

C:\Users\tipto\AppData\Local\Temp\ipykernel_33776\3331607960.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [37]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.161700,0.154407,0.713136,0.759612,0.735641,0.954412
2,0.121800,0.124080,0.768944,0.796631,0.782543,0.963414
3,0.097500,0.119202,0.784614,0.800678,0.792565,0.965343


TrainOutput(global_step=6297, training_loss=0.1682113425133814, metrics={'train_runtime': 428.3616, 'train_samples_per_second': 235.112, 'train_steps_per_second': 14.7, 'total_flos': 2363814501589146.0, 'train_loss': 0.1682113425133814, 'epoch': 3.0})

In [38]:
# evaluate the model on test data
trainer.evaluate(
    eval_dataset = tokenized_dataset['test']
)

{'eval_loss': 0.12392827123403549,
 'eval_precision': 0.7848628630945492,
 'eval_recall': 0.8022237993849065,
 'eval_f1': 0.7934483767183387,
 'eval_accuracy': 0.9645709852204217,
 'eval_runtime': 9.9778,
 'eval_samples_per_second': 720.999,
 'eval_steps_per_second': 45.1,
 'epoch': 3.0}

In [39]:
trainer.save_model()

In [40]:
import json 
save_path = "./ner-bert-model"

In [41]:
# save the model
model.save_pretrained(save_path)

In [42]:
# save the tokenizer
tokenizer.save_pretrained(save_path)

('./ner-bert-model\\tokenizer_config.json',
 './ner-bert-model\\special_tokens_map.json',
 './ner-bert-model\\vocab.txt',
 './ner-bert-model\\added_tokens.json',
 './ner-bert-model\\tokenizer.json')

In [44]:
# Save label mappings
label_mapping = {
    "label2id": label2id,
    "id2label": {int(k): v for k, v in id2label.items()}  
}

In [45]:
with open(f"{save_path}/label_mapping.json", "w") as f:
    json.dump(label_mapping, f, indent = 2)

In [46]:
from transformers import AutoTokenizer

In [48]:
# complete loading function
def load_ner_model(model_path):
    # load model
    model = AutoModelForTokenClassification.from_pretrained(model_path)
    # load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    # load label mapping
    try:
        with open(f"{model_path}/label_mapping.json", "r") as f:
            label_mapping = json.load(f)
            label2id = label_mapping["label2id"]
            id2label = {int(k): v for k, v in label_mapping["id2label"].items()}
    except FileNotFoundError:
        label2id = model.config.label2id
        id2label = model.config.id2label
        
        
    return model, tokenizer, label2id, id2label

In [49]:
model, tokenizer, label2id, id2label = load_ner_model(save_path)

In [50]:
import torch

In [57]:
def predict_ner(text, model, tokenizer, id2label):
    model.eval()
    
    # tokenize the input text
    inputs = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=512
        # no need of padding cause we are doing batch size 1 here
    )
    
    # get prediction
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=2)  # ← Fixed: access .logits
    
    # convert tokens and predictions to list
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    predicted_labels = [id2label[p.item()] for p in predictions[0]]
    
    entities = []
    current_entity = None
    
    for token, label in zip(tokens, predicted_labels):
        # Skip special tokens
        if token in ['[CLS]', '[SEP]', '[PAD]']:
            continue
        # Handle entity labels
        if label.startswith('B-'):  # Beginning of entity
            # Save previous entity if exists
            if current_entity:
                entities.append(current_entity)
            
            # Start new entity
            entity_type = label[2:]  # Remove 'B-' prefix
            current_entity = {
                'entity': token.replace('##', ''),  # Remove subword marker
                'type': entity_type,
                'label': label
            }
        
        elif label.startswith('I-') and current_entity:  # Inside entity
            # Continue current entity
            if token.startswith('##'):
                current_entity['entity'] += token[2:]  # Remove ## and append
            else:
                current_entity['entity'] += ' ' + token
        
        elif label == 'O':  # Outside any entity
            # Save current entity if exists
            if current_entity:
                entities.append(current_entity)
                current_entity = None
    
    # Don't forget the last entity
    if current_entity:
        entities.append(current_entity)
    
    return entities  

In [58]:
text = """
Apple Inc. is planning to open a new store in New York City. 
The CEO Tim Cook announced this during a conference in San Francisco. 
The company, founded by Steve Jobs, continues to expand globally.
"""

In [59]:
entities = predict_ner(text, model, tokenizer, id2label)

In [61]:
for entity in entities:
    print(f"  {entity['entity']} -> {entity['type']}")

  Apple Inc -> org
  New York City . The CEO Tim Cook -> geo
  San Francisco -> geo
  Steve -> per
